In [1]:
import pandas as pd
import os
import polars as pl

In [2]:
from collections import defaultdict

In [3]:
import cobra

In [4]:
from cobra.io import read_sbml_model

In [5]:
proj_dir = "/home/users/gsantamaria/projects/metabognn_dev/"
proj_dir_scratch = "/scratch/users/gsantamaria/test_large_files/metabognn_dev/"

In [6]:
attene_f = os.path.join(
    proj_dir,
    "data/validation/attene_ramos/AID_720637_datatable.csv"
)
attene_drugs_f = os.path.join(
    proj_dir_scratch,
    "results/mc_phenotype_assoc/HEPG2/attene_mmp_down_drugs.tsv"
)

In [7]:
mc_dir = os.path.join(
    proj_dir_scratch, "results/metChange/HEPG2/rank_concat"
)

mc_f = os.path.join(mc_dir, "expr_ensembl_rank_metChange_rpmi_medium_concat.csv")

In [8]:
expr_dir = os.path.join(
    proj_dir_scratch, "data/perturbation/lincs/cell_matrices/HEPG2"
)
expr_f = os.path.join(expr_dir, "expr_ensembl.csv")
pnlt_f = os.path.join(expr_dir, "expr_ensembl_rank.csv")
mdat_f = os.path.join(expr_dir, "mdat.csv")

In [9]:
gem_f = os.path.join(proj_dir, "data/GEMs/Human-GEM/model/Human-GEM.xml")

In [10]:
gem = read_sbml_model(gem_f)

In [11]:
gem.reactions[24]

Reaction identifier,MAR00033
Name,
Memory address,0x7f1daa986810
Stoichiometry,"MAM00611c + MAM00611m + MAM00622c + MAM00622m --> MAM03140c 24-oxo-1alpha,23,25-trihydroxyvitamin D3 + 24-oxo-1alpha,23,25-trihydroxyvitamin D3 + 25-hydroxyvitamin D3-26,23-lactone + 25-hydroxyvitamin D3-26,23-lactone --> vitamin D derivatives"
GPR,
Lower bound,0.0
Upper bound,1000.0


In [12]:
# Get unique subsystems to see what name does TCA cycle hold
uniq_subsysts = []
for rxn in gem.reactions:
    rxn_subsysts = rxn.subsystem
    if isinstance(rxn_subsysts, str):
        if rxn_subsysts not in uniq_subsysts:
            uniq_subsysts.append(rxn_subsysts)
    else:
        for subs in rxn_subsysts:
            if subs not in uniq_subsysts:
                uniq_subsysts.append(subs)

In [16]:
def get_metrxndict(gem, whatmets="products", subsystem="all"):
    metrxndict = defaultdict(list)
    for r in gem.reactions:
        if subsystem == "all":
            keep_rxn = True
        else:
            keep_rxn = r.subsystem == subsystem
        if keep_rxn:
            is_rev = (r.lower_bound < 0 and r.upper_bound > 0) or (r.lower_bound > 0 and r.upper_bound < 0)
            if is_rev:
                mets = r.metabolites
            else:
                mets = getattr(r, whatmets)
            for m in mets:
                if r.id not in metrxndict[m.id]:
                    metrxndict[m.id].append(r.id)
    metrxndict = dict(metrxndict)
    return metrxndict

In [17]:
prod2rxn_tca = get_metrxndict(gem, "products", subsystem='Tricarboxylic acid cycle and glyoxylate/dicarboxylate metabolism')
reac2rxn_tca = get_metrxndict(gem, "reactants", subsystem='Tricarboxylic acid cycle and glyoxylate/dicarboxylate metabolism')

In [18]:
list(gem.reactions[100].genes)[0].id

'ENSG00000147383'

In [19]:
def parse_metrnxndict(metrxndict, gem=gem):
    ids = []
    names = []
    formulas = []
    compartments = []
    reactions = []
    genes = []
    for met in list(metrxndict.keys()):
        m = gem.metabolites.get_by_id(met)
        ids.append(m.id)
        names.append(m.name)
        formulas.append(m.formula)
        compartments.append(m.formula)
        rxns = metrxndict[met]
        reactions.append('|'.join(rxns))
        g = []
        for r in rxns:
            for gene in list(gem.reactions.get_by_id(r).genes):
                g.append(gene.id)
        g = '|'.join(g)     
        genes.append(g)
    df = pd.DataFrame({
        "id": ids,
        "name": names,
        "formula": formulas,
        "compartment": compartments,
        "reactions": reactions,
        "genes": genes
    })
    return df

In [20]:
prod2rxn_tca_df = parse_metrnxndict(prod2rxn_tca, gem=gem)
reac2rxn_tca_df = parse_metrnxndict(reac2rxn_tca, gem=gem)

In [21]:
prod2rxn_tca_df

,id,name,formula,compartment,reactions,genes
0,MAM02183c,isocitrate,C6H5O7,C6H5O7,MAR00710|MAR04454,ENSG00000138413|ENSG00000122729
1,MAM02554c,NADP+,C21H25N7O17P3,C21H25N7O17P3,MAR00710|MAR04465,ENSG00000138413|ENSG00000137106
2,MAM01306c,AKG,C5H4O5,C5H4O5,MAR00710,ENSG00000138413
3,MAM01596c,CO2,CO2,CO2,MAR00710|MAR08774,ENSG00000138413
4,MAM02555c,NADPH,C21H26N7O17P3,C21H26N7O17P3,MAR00710,ENSG00000138413
...,...,...,...,...,...,...
58,MAM00674c,2-phospho-D-glycerate,C3H4O7P,C3H4O7P,MAR08781,
59,MAM02439x,malate,C4H4O5,C4H4O5,MAR08981,
60,MAM02552x,NAD+,C21H26N7O14P2,C21H26N7O14P2,MAR08981,
61,MAM02553x,NADH,C21H27N7O14P2,C21H27N7O14P2,MAR08981,


In [22]:
attene_drugs = pd.read_csv(attene_drugs_f, sep = '\t')

In [23]:
(attene_drugs["labels_seen"] == "MMP_down").sum()

np.int64(104)

In [24]:
attene_drugs_down = attene_drugs[attene_drugs["group"] == "MMP_down"]

In [25]:
mdat = pd.read_csv(mdat_f)

In [26]:
attene_drugs_down

,drug,group,n_lincs_perts,n_records,labels_seen,potency_uM,efficacy_pct
0,AM-580,MMP_down,1,1,MMP_down,58.8300,-121.30
2,BRD-A29731977,MMP_down,1,2,MMP_down;cytotox,11.4400,-62.20
5,BRD-A66861218,MMP_down,1,1,MMP_down,28.7300,-95.85
8,BRD-K01815685,MMP_down,12,1,MMP_down,43.6600,-102.80
9,BRD-K09784055,MMP_down,6,2,MMP_down;incon,24.3400,-74.43
...,...,...,...,...,...,...,...
587,triciribine,MMP_down,7,1,MMP_down,1.2060,-62.57
588,triclosan,MMP_down,7,2,MMP_down,1.3770,-102.90
592,troglitazone,MMP_down,9,2,MMP_down,10.5900,-121.70
605,vinblastine,MMP_down,9,2,MMP_down,0.8873,-42.03


In [27]:
mdat["pertname"]

0          vorinostat
1           lapatinib
2        mitoxantrone
3         fulvestrant
4             HMN-214
             ...     
51450            DMSO
51451            DMSO
51452            DMSO
51453            DMSO
51454            DMSO
Name: pertname, Length: 51455, dtype: object

In [28]:
keep_perts = [x in attene_drugs_down["drug"].to_list() or x == "DMSO" for x in mdat["pertname"].to_list()]

In [29]:
keep_ids = mdat.loc[keep_perts, "id"].to_list()

In [32]:
cols_mc = pl.scan_csv(mc_f).collect_schema().names()
mc = (
    pl.scan_csv(mc_f)
    .select([cols_mc[0]] + keep_ids)
    .collect()
)

In [33]:
cols_expr = pl.scan_csv(expr_f).collect_schema().names()

In [34]:
expr = (
    pl.scan_csv(expr_f)
    .select([cols_expr[0]] + keep_ids)
    .collect()
)

In [35]:
expr

ensembl_gene_id,ABY001_HEPG2_XH_X1_B15:N11,ABY001_HEPG2_XH_X1_B15:M03,ABY001_HEPG2_XH_X1_B15:D19,ABY001_HEPG2_XH_X1_B15:L11,ABY001_HEPG2_XH_X1_B15:D15,ABY001_HEPG2_XH_X1_B15:E15,ABY001_HEPG2_XH_X1_B15:N07,ABY001_HEPG2_XH_X1_B15:E23,ABY001_HEPG2_XH_X1_B15:L07,ABY001_HEPG2_XH_X1_B15:E19,ABY001_HEPG2_XH_X1_B15:L03,ABY001_HEPG2_XH_X1_B15:N03,ABY001_HEPG2_XH_X1_B15:M11,ABY001_HEPG2_XH_X1_B15:M07,ABY001_HEPG2_XH_X1_B15:F15,ABY001_HEPG2_XH_X1_B15:F23,ABY001_HEPG2_XH_X1_B15:D23,CPC004_HEPG2_6H_X1_B5_DUO52HI53LO:F03,CPC004_HEPG2_6H_X3.L2_F2B5_DUO52HI53LO:D09,CPC004_HEPG2_6H_X3.L2_F2B5_DUO52HI53LO:G11,CPC004_HEPG2_6H_X3.L2_F2B5_DUO52HI53LO:I14,CPC004_HEPG2_6H_X1_B5_DUO52HI53LO:K09,CPC004_HEPG2_6H_X2_F1B5_DUO52HI53LO:E14,CPC004_HEPG2_6H_X1_B5_DUO52HI53LO:I14,CPC004_HEPG2_6H_X3.L2_F2B5_DUO52HI53LO:K11,CPC004_HEPG2_6H_X2_F1B5_DUO52HI53LO:D12,CPC004_HEPG2_6H_X1_B5_DUO52HI53LO:D12,CPC004_HEPG2_6H_X1_B5_DUO52HI53LO:G09,CPC004_HEPG2_6H_X1_B5_DUO52HI53LO:L04,CPC004_HEPG2_6H_X3.L2_F2B5_DUO52HI53LO:I07,CPC004_HEPG2_6H_X1_B5_DUO52HI53LO:J13,CPC004_HEPG2_6H_X2_F1B5_DUO52HI53LO:E07,CPC004_HEPG2_6H_X1_B5_DUO52HI53LO:L07,CPC004_HEPG2_6H_X3.L2_F2B5_DUO52HI53LO:F01,CPC004_HEPG2_6H_X1_B5_DUO52HI53LO:H08,CPC004_HEPG2_6H_X1_B5_DUO52HI53LO:E23,…,POL001_HEPG2_24H_X2.L2_B23:B06,POL001_HEPG2_24H_X1.L2_B23:J16,POL001_HEPG2_24H_X2.L2_B23:A04,POL001_HEPG2_24H_X2.L2_B23:J17,POL001_HEPG2_24H_X2.L2_B23:B04,POL001_HEPG2_24H_X3.L2_B23:J16,POL001_HEPG2_24H_X2.L2_B23:N24,POL001_HEPG2_24H_X2.L2_B23:F07,POL001_HEPG2_24H_X3.L2_B23:B03,POL001_HEPG2_24H_X2.L2_B23:F10,POL001_HEPG2_24H_X3.L2_B23:F08,POL001_HEPG2_24H_X2.L2_B23:F08,POL001_HEPG2_24H_X2.L2_B23:F09,POL001_HEPG2_24H_X3.L2_B23:J14,POL001_HEPG2_24H_X3.L2_B23:F07,POL001_HEPG2_24H_X3.L2_B23:F12,POL001_HEPG2_24H_X3.L2_B23:A03,POL001_HEPG2_24H_X2.L2_B23:J13,POL001_HEPG2_24H_X3.L2_B23:N24,POL001_HEPG2_24H_X1.L2_B23:J13,POL001_HEPG2_24H_X3.L2_B23:N21,POL001_HEPG2_24H_X3.L2_B23:A06,POL001_HEPG2_24H_X2.L2_B23:N23,POL001_HEPG2_24H_X1.L2_B23:B03,POL001_HEPG2_24H_X2.L2_B23:J18,POL001_HEPG2_24H_X2.L2_B23:J14,POL001_HEPG2_24H_X1.L2_B23:B06,POL001_HEPG2_24H_X2.L2_B23:J15,POL001_HEPG2_24H_X1.L2_B23:F12,POL001_HEPG2_24H_X1.L2_B23:B04,POL001_HEPG2_24H_X1.L2_B23:N21,POL001_HEPG2_24H_X3.L2_B23:F10,POL001_HEPG2_24H_X1.L2_B23:F08,POL001_HEPG2_24H_X1.L2_B23:A06,POL001_HEPG2_24H_X3.L2_B23:J18,POL001_HEPG2_24H_X3.L2_B23:B04,POL001_HEPG2_24H_X3.L2_B23:J17
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""ENSG00000121410""",4.259391,3.45476,3.75795,3.67487,4.375798,3.498071,3.253668,4.066302,4.259429,3.422022,3.174023,3.698892,3.332827,2.984651,3.111757,3.715392,3.948816,7.184917,4.098114,4.167649,5.319264,6.326293,6.697548,5.863348,6.15738,5.787116,8.092418,7.180867,7.229209,4.4642,8.170682,6.284379,7.406728,3.716609,7.517229,4.663937,…,2.672227,2.967208,2.779544,3.022542,2.918657,2.875971,2.821694,3.179074,2.703044,2.640418,2.777147,3.089819,2.49099,2.732576,2.959515,3.007277,2.565651,2.852891,2.888047,2.65674,3.184114,2.795784,2.815416,2.944061,2.772785,2.81195,2.942914,2.882027,3.285295,2.860934,3.047303,2.70038,2.835562,3.094146,2.596588,2.716797,2.886917
"""ENSG00000148584""",5.02895,5.701091,4.588117,5.62955,4.31662,4.908123,4.921504,5.323095,5.173924,4.887266,3.928919,5.655976,5.012664,5.132788,4.219748,5.365309,5.30384,6.560016,4.750107,4.659698,5.20681,5.098155,5.018672,5.778746,4.435904,5.127375,6.219457,6.31927,6.243583,3.965511,6.198976,3.887662,6.101072,4.635123,5.606835,6.208144,…,4.974272,4.987451,5.17595,4.931371,4.765587,4.958194,4.933106,4.743725,5.000041,4.467065,4.423732,5.023893,4.268767,4.845041,4.68034,5.34398,4.788734,4.665911,4.924009,4.783952,5.243553,4.715166,5.187104,5.26611,4.791726,4.16482,4.676994,5.125788,4.719841,4.58,5.071605,4.660334,4.420251,5.365908,4.593176,5

# Where does the sign come from?

**The observation.** In HepG2, `tca_endpoints` MetChange `log2FC` separates MMP-down drugs from
inactive ones at AUROC **0.357–0.366** (random-set p 0.005). AUROC < 0.5 means MMP-down drugs sit
**lower**, and since the MetChange score is a cost (`run_metchange_exe.py` line 266: *"a metchange
score of zero means the metabolite can be easily produced"*), lower = **easier to produce**. So
drugs that depolarise mitochondria appear to make the TCA intermediates *cheaper*, which is the
opposite of what "mitochondrial damage impairs the TCA cycle" predicts.

**Four places the sign could be set.** Each layer is a transformation of the one above, so the flip
has to happen at exactly one of them. The point of this notebook is to find which.

| layer | file | what it holds | sign convention |
|---|---|---|---|
| 1. expression | `expr_ensembl.csv` | absolute expression per sample | higher = more transcript |
| 2. penalty | `expr_ensembl_rank.csv` | `1 - rank/n` within each sample, top 2% forced to 1 | **higher = worse** (less expressed) |
| 3. LP | `..._metChange_rpmi_medium_concat.csv` | MetChange score per metabolite per sample | **higher = harder to produce** |
| 4. limma | `results/labels/HEPG2/metchange/rank/log2FC.csv` | drug vs batch/timepoint-matched controls | difference of layer 3 |

### What each possible flip would mean

**Layer 1 → 2 is the one to look at first, and it is NOT a fixed inversion.** Within a single sample
a gene with higher expression necessarily has a lower penalty. But *across* samples the two can move
the same way: the penalty is a **rank within its own sample**, while absolute expression is not. If a
drug suppresses the producer genes by 0.3 units and the median gene by 0.6, those genes fall in
absolute expression yet **rise in rank**, so their penalties fall too. Layers 1 and 2 landing on the
**same** side of 0.5 is therefore the signature of **rank-relativity**, and it would mean the
expression and MetChange arms were never in conflict — one reports a level, the other a rank.

**Layer 2 → 3** flipping means the LP is not following its own inputs: penalties got worse yet the
metabolites got cheaper. The likely cause is that the LP takes a **minimum over routes**, so the
cheapest route to a TCA metabolite need not run through the genes that deteriorated. This is also
consistent with the producer-average arm (which takes the *mean* over producers) being null where
MetChange is not.

**Layer 3 → 4** flipping means **limma** sets the sign: the raw pooled difference and the fitted
coefficient disagree, i.e. the pooled difference is confounded with batch or timepoint and the
fitted coefficient is the one to trust. The batch-balance check below is what to read in that case.

**No flip anywhere** would mean the puzzle is an artefact of the gene set used for the expression
comparison — see the next paragraph.

### On the gene set

The expression arm in reports 09–11 scores `tca gene`, the GPR genes of reactions annotated to the
Human-GEM subsystem `Tricarboxylic acid cycle and glyoxylate/dicarboxylate metabolism`. That is a
merged KEGG annotation and is **under half citrate cycle**: of its 45 genes ~20 are canonical, and
the rest include **7 LDH isoforms, 3 aldolases (ALDOA/B/C) and 8 ALDH isoforms** — glycolytic,
highly drug-responsive genes. The metabolite side has a clean alternative (`tca_endpoints`); the gene
side has none. So this notebook uses **producer genes** — the GPR genes of the reactions that can
actually produce each TCA metabolite — and reports the subsystem set only for comparison.


In [36]:
#  Set definitions, copied verbatim from scripts/mc_phenotype_assoc/build_metabolite_sets.py
#  so this notebook reproduces the sets the reports used rather than approximating them.

# CURATED['tca_endpoints']. CO2 is in the list but is a currency name, so it is dropped
# from every *_nohub variant - which is why the nohub sets resolve to 6 names, not 7.
TCA_ENDPOINTS = ['acetyl-CoA', 'citrate', 'AKG', 'succinate', 'fumarate', 'malate', 'CO2']

# SUBSYSTEMS['tca'] - the merged KEGG annotation, kept only as a comparison
TCA_SUBSYSTEM = 'Tricarboxylic acid cycle and glyoxylate/dicarboxylate metabolism'

# CURRENCY_NAMES: hubs by name, on top of whatever the degree cutoff catches
CURRENCY_NAMES = {'ATP', 'ADP', 'AMP', 'NAD+', 'NADH', 'NADP+', 'NADPH', 'H+', 'H2O',
                  'Pi', 'PPi', 'CoA', 'O2', 'CO2', 'NH3', 'HCO3-', 'FAD', 'FADH2',
                  'GTP', 'GDP', 'UTP', 'UDP', 'S-adenosyl-L-methionine'}

HUB_PERCENTILE = 99.0   # --hub-percentile default
COMPARTMENTS = {'cyto': {'c'}, 'mito': {'m'}}   # 'all' = no filter


In [37]:
import numpy as np

#  Hub flag: degree = number of reactions a metabolite takes part in, cutoff at the
#  99th percentile, PLUS anything named in CURRENCY_NAMES.
degree = defaultdict(int)
for r in gem.reactions:
    for m in r.metabolites:
        degree[m.id] += 1
hub_cut = float(np.percentile(list(degree.values()), HUB_PERCENTILE))
is_hub = {m.id: (degree[m.id] >= hub_cut or m.name in CURRENCY_NAMES) for m in gem.metabolites}
print('hub cutoff: degree >= %.0f  (%d hubs of %d metabolites)'
      % (hub_cut, sum(is_hub.values()), len(is_hub)))


def met_set(names=None, subsystem=None, variant='all_nohub'):
    """Metabolite ids for one set/variant, exactly as build_metabolite_sets.py defines them.

    `names`     -> a CURATED set: every id whose metabolite NAME is in the list.
    `subsystem` -> a SUBSYSTEMS set: every metabolite taking part in a reaction of it.
    `variant`   -> 'all' | 'cyto' | 'mito', optionally suffixed '_nohub'.
    """
    if (names is None) == (subsystem is None):
        raise ValueError('give exactly one of names / subsystem')
    if names is not None:
        want = set(names)
        ids = {m.id for m in gem.metabolites if m.name in want}
    else:
        ids = set()
        for r in gem.reactions:
            if r.subsystem == subsystem:
                ids |= {m.id for m in r.metabolites}
    base, nohub = (variant[:-6], True) if variant.endswith('_nohub') else (variant, False)
    if base in COMPARTMENTS:
        ids = {i for i in ids if gem.metabolites.get_by_id(i).compartment in COMPARTMENTS[base]}
    if nohub:
        ids = {i for i in ids if not is_hub.get(i, False)}
    return sorted(ids)


#  The set the result is reported on, and the contaminated subsystem set for comparison
tca_end = {v: met_set(names=TCA_ENDPOINTS, variant=v)
           for v in ['all', 'all_nohub', 'cyto_nohub', 'mito_nohub']}
tca_sub = {v: met_set(subsystem=TCA_SUBSYSTEM, variant=v)
           for v in ['all_nohub', 'mito_nohub']}

for lab, d in [('tca_endpoints', tca_end), ('tca (subsystem)', tca_sub)]:
    for v, ids in d.items():
        names = sorted({gem.metabolites.get_by_id(i).name for i in ids})
        print('%-16s %-11s %3d ids, %2d names: %s'
              % (lab, v, len(ids), len(names), ', '.join(names)))


hub cutoff: degree >= 75  (183 hubs of 8461 metabolites)
tca_endpoints    all          33 ids,  7 names: AKG, CO2, acetyl-CoA, citrate, fumarate, malate, succinate
tca_endpoints    all_nohub    24 ids,  6 names: AKG, acetyl-CoA, citrate, fumarate, malate, succinate
tca_endpoints    cyto_nohub    6 ids,  6 names: AKG, acetyl-CoA, citrate, fumarate, malate, succinate
tca_endpoints    mito_nohub    5 ids,  5 names: AKG, citrate, fumarate, malate, succinate
tca (subsystem)  all_nohub    45 ids, 30 names: (S)-Glycerate, 2-phospho-D-glycerate, 3-carboxy-1-hydroxypropyl-ThPP, AKG, D-xylulose, D-xylulose-1-phosphate, DHAP, OAA, S-succinyldihydrolipoamide, acetoacetate, acetoacetyl-CoA, acetyl-CoA, cis-aconitate, citrate, dihydrolipoamide, fumarate, glycerate, glycolaldehyde, glycolate, glyoxalate, hydroxypyruvate, isocitrate, lipoamide, malate, oxalate, succinate, succinyl-CoA, thiamin-PP, ubiquinol, ubiquinone
tca (subsystem)  mito_nohub   22 ids, 22 names: 3-carboxy-1-hydroxypropyl-ThPP, AKG

In [38]:
#  PRODUCER GENES, matching scripts/mc_phenotype_assoc/build_derived_labels.py::producer_genes
#
#  A reaction can PRODUCE metabolite m if m is on the product side and the reaction can
#  carry forward flux, or on the substrate side and it can carry reverse flux. Direction
#  is read from the bounds, not from `r.products` / `r.reactants`, because cobra assigns
#  those by stoichiometric sign alone and so mislabels a reverse-only reaction.
#
#  Reactions with an empty GPR (exchanges, diffusion, sinks) contribute no genes. That is
#  correct here - they carry no expression information - but note it is also why import is
#  expression-independent: a GPR-less reaction is scored at the sample's MEDIAN penalty
#  (run_metchange_exe.py, `results[rxn_id] = median_arr.copy()`).
#
#  NB two bugs in the draft `get_metrxndict` / `parse_metrnxndict` above, if you reuse them:
#    * `metrxndict[p.id]` should be `metrxndict[m.id]` - `p` is undefined
#    * `compartments.append(m.formula)` should be `m.compartment`

def producers_of(met_ids):
    """metabolite id -> {Ensembl gene id} for reactions that can produce it."""
    out = {m: set() for m in met_ids}
    want = set(met_ids)
    for r in gem.reactions:
        genes = {g.id for g in r.genes}
        if not genes:
            continue
        fwd, rev = r.upper_bound > 0, r.lower_bound < 0
        for m, coef in r.metabolites.items():
            if m.id in want and ((coef > 0 and fwd) or (coef < 0 and rev)):
                out[m.id] |= genes
    return out


def subsystem_genes(subsystem):
    """GPR genes of every reaction annotated to one subsystem = the `tca gene` set."""
    g = set()
    for r in gem.reactions:
        if r.subsystem == subsystem:
            g |= {x.id for x in r.genes}
    return g


prod_genes = {v: producers_of(ids) for v, ids in tca_end.items()}
PROD_SET = {v: set().union(*d.values()) if d else set() for v, d in prod_genes.items()}
SUB_SET = subsystem_genes(TCA_SUBSYSTEM)

sym = {g.id: (g.name or g.id) for g in gem.genes}
for v in ['all_nohub', 'mito_nohub', 'cyto_nohub']:
    print('tca_endpoints/%-11s %3d metabolites -> %4d producer genes'
          % (v, len(tca_end[v]), len(PROD_SET[v])))
print()
print('`tca gene` subsystem set: %d genes' % len(SUB_SET))
shared = PROD_SET['all_nohub'] & SUB_SET
print('shared with producers(all_nohub): %d  -> %.0f%% of producers, %.0f%% of the subsystem set'
      % (len(shared), 100 * len(shared) / max(len(PROD_SET['all_nohub']), 1),
         100 * len(shared) / max(len(SUB_SET), 1)))
print()
print('per metabolite:')
for m in tca_end['all_nohub']:
    print('   %-12s %-14s %3d producer genes'
          % (m, gem.metabolites.get_by_id(m).name, len(prod_genes['all_nohub'][m])))


tca_endpoints/all_nohub    24 metabolites ->  173 producer genes
tca_endpoints/mito_nohub    5 metabolites ->   36 producer genes
tca_endpoints/cyto_nohub    6 metabolites ->  132 producer genes

`tca gene` subsystem set: 45 genes
shared with producers(all_nohub): 22  -> 13% of producers, 49% of the subsystem set

per metabolite:
   MAM01261c    acetyl-CoA      86 producer genes
   MAM01261g    acetyl-CoA       1 producer genes
   MAM01261n    acetyl-CoA      34 producer genes
   MAM01261r    acetyl-CoA       2 producer genes
   MAM01306c    AKG             26 producer genes
   MAM01306e    AKG             11 producer genes
   MAM01306m    AKG             18 producer genes
   MAM01306r    AKG              0 producer genes
   MAM01306x    AKG              2 producer genes
   MAM01587c    citrate          5 producer genes
   MAM01587e    citrate          0 producer genes
   MAM01587m    citrate          3 producer genes
   MAM01862c    fumarate         7 producer genes
   MAM01862e    fu

In [39]:
#  Sample groups. `keep_ids` above already holds the MMP_down samples plus every DMSO;
#  split it, and record batch/timepoint so the pooled and the matched comparison can be
#  run separately. The matched one matters: limma fits each perturbation against "its own
#  samples and the controls of the same timepoint in its batches" (limma_DE.R), so a
#  pooled difference and a limma coefficient are NOT the same quantity, and layer 3 vs 4
#  is exactly that comparison.

print('mdat columns:', list(mdat.columns))
print()

down_drugs = set(attene_drugs_down['drug'])
mdat_keep = mdat.loc[keep_perts].copy()
mdat_keep['arm'] = np.where(mdat_keep['pertname'] == 'DMSO', 'ctrl', 'case')

case_ids = mdat_keep.loc[mdat_keep['arm'] == 'case', 'id'].tolist()
ctrl_ids = mdat_keep.loc[mdat_keep['arm'] == 'ctrl', 'id'].tolist()
print('case samples (MMP_down drugs): %d   from %d distinct drugs'
      % (len(case_ids), mdat_keep.loc[mdat_keep.arm == 'case', 'pertname'].nunique()))
print('ctrl samples (DMSO)          : %d' % len(ctrl_ids))
print()

#  Batch/timepoint composition. If the two arms are distributed differently across these,
#  a pooled difference is confounded and limma would be right to disagree with it.
for col in [c for c in ('rna_plate', 'batch', 'pert_time', 'time', 'det_plate') if c in mdat.columns]:
    ct = mdat_keep.groupby(['arm', col]).size().unstack(fill_value=0)
    frac_case = ct.loc['case'] / max(ct.loc['case'].sum(), 1)
    frac_ctrl = ct.loc['ctrl'] / max(ct.loc['ctrl'].sum(), 1)
    tvd = 0.5 * float(np.abs(frac_case - frac_ctrl).sum())   # total variation distance
    print('%-12s %3d levels   total-variation distance between arms: %.3f%s'
          % (col, ct.shape[1], tvd, '   <-- badly unbalanced' if tvd > 0.3 else ''))


mdat columns: ['V1', 'cell', 'dose', 'id', 'inchi_key', 'pert_aliases', 'pert_id', 'pertname', 'smiles', 'timepoint', 'batch', 'exper_group', 'cell_type_fine', 'perturbation']

case samples (MMP_down drugs): 2587   from 144 distinct drugs
ctrl samples (DMSO)          : 2895

batch         52 levels   total-variation distance between arms: 0.457   <-- badly unbalanced


In [40]:
#  Load the three sample-level matrices, restricted to the samples of interest.
#  Row-id column names differ between files, so take whatever the first column is called.

def load_cols(path, ids):
    """polars scan -> pandas, first column as index, only `ids` kept (in file order)."""
    names = pl.scan_csv(path).collect_schema().names()
    keep = [c for c in ids if c in set(names)]
    missing = [c for c in ids if c not in set(names)]
    df = (pl.scan_csv(path).select([names[0]] + keep).collect()).to_pandas()
    df = df.set_index(names[0])
    print('%-24s %6d rows x %5d samples kept (%d of the requested ids absent)'
          % (os.path.basename(path), df.shape[0], df.shape[1], len(missing)))
    return df


EXPR = load_cols(expr_f, case_ids + ctrl_ids)    # layer 1: absolute expression
PNLT = load_cols(pnlt_f, case_ids + ctrl_ids)    # layer 2: rank penalties
MC   = load_cols(mc_f,   case_ids + ctrl_ids)    # layer 3: raw MetChange scores

#  Sanity-check layer 2 really is the penalty and not the raw rank: p = 1 - rank/n, so it
#  must sit in [0, 1], and ~2% of values must be exactly 1 (the --threshold 2 ceiling).
v = PNLT.to_numpy(dtype=float).ravel()
v = v[np.isfinite(v)]
print()
print('penalty matrix: min %.4f  max %.4f  median %.4f  fraction exactly 1.0: %.3f%%'
      % (v.min(), v.max(), np.median(v), 100 * np.mean(v == 1.0)))
print('  expected if these are penalties: range [0,1], median ~0.5, ~2%% at the 1.0 ceiling')
print()
print('MetChange matrix: %d features; index looks like %s'
      % (MC.shape[0], list(MC.index[:3])))
print('  a MetChange score of 0 means the metabolite is EASY to produce (cost, not capacity)')


expr_ensembl.csv          20587 rows x  5482 samples kept (0 of the requested ids absent)
expr_ensembl_rank.csv      2794 rows x  5482 samples kept (0 of the requested ids absent)
expr_ensembl_rank_metChange_rpmi_medium_concat.csv   5707 rows x  5482 samples kept (0 of the requested ids absent)

penalty matrix: min 0.0000  max 1.0000  median 0.3355  fraction exactly 1.0: 4.802%
  expected if these are penalties: range [0,1], median ~0.5, ~2%% at the 1.0 ceiling

MetChange matrix: 5707 features; index looks like ['MAM00003c', 'MAM00003e', 'MAM00003l']
  a MetChange score of 0 means the metabolite is EASY to produce (cost, not capacity)


In [41]:
from scipy.stats import rankdata

#  AUROC = P(a random CASE value > a random CTRL value). Below 0.5 means cases sit LOWER.
#  Computed from ranks so it is the exact Mann-Whitney statistic, ties included.

def auroc(case_vals, ctrl_vals):
    a = np.asarray(case_vals, float)
    b = np.asarray(ctrl_vals, float)
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    if len(a) < 3 or len(b) < 3:
        return np.nan, len(a), len(b)
    r = rankdata(np.concatenate([a, b]))
    u = r[:len(a)].sum() - len(a) * (len(a) + 1) / 2.0
    return u / (len(a) * len(b)), len(a), len(b)


def set_scores(mat, members, how='z'):
    """One value per SAMPLE for a set of features.

    how='raw' : plain mean over the set members - the absolute level
    how='z'   : (set mean - that sample's mean over ALL features) / that sample's SD.
                This is the statistic mc_phenotype_assoc.py reports, so it is the one the
                AUROC 0.357 refers to. It removes each sample's global level, which also
                makes it insensitive to the dominant global component of report 11 section 8.

    The raw/z contrast is itself diagnostic: a set that moves on `z` but not on `raw` moved
    only RELATIVE to the rest of the matrix.
    """
    rows = [m for m in members if m in mat.index]
    if not rows:
        return None, 0
    sub = mat.loc[rows].to_numpy(dtype=float)
    s = np.nanmean(sub, axis=0)
    if how == 'raw':
        return pd.Series(s, index=mat.columns), len(rows)
    allv = mat.to_numpy(dtype=float)
    mu = np.nanmean(allv, axis=0)
    sd = np.nanstd(allv, axis=0)
    with np.errstate(invalid='ignore', divide='ignore'):
        z = (s - mu) / np.where(sd > 0, sd, np.nan)
    return pd.Series(z, index=mat.columns), len(rows)


VARIANT = 'all_nohub'          # switch to 'mito_nohub' / 'cyto_nohub' to repeat
genes = sorted(PROD_SET[VARIANT])
mets = tca_end[VARIANT]

ladder = []
for layer, mat, members, meaning in [
        ('1 expression', EXPR, genes, 'higher = more transcript'),
        ('2 penalty',    PNLT, genes, 'higher = WORSE (less expressed)'),
        ('3 MetChange',  MC,   mets,  'higher = HARDER to produce'),
]:
    for how in ('raw', 'z'):
        sc, n = set_scores(mat, members, how=how)
        if sc is None:
            ladder.append((layer, how, np.nan, 0, 0, 0, meaning))
            continue
        a, na, nb = auroc(sc.reindex(case_ids).dropna(), sc.reindex(ctrl_ids).dropna())
        ladder.append((layer, how, a, n, na, nb, meaning))

print('MMP_down cases vs DMSO controls, tca_endpoints/%s' % VARIANT)
print('%d producer genes, %d metabolites' % (len(genes), len(mets)))
print()
print('%-13s %-5s %8s %7s %7s %7s  %s'
      % ('layer', 'stat', 'AUROC', 'nfeat', 'ncase', 'nctrl', 'higher value means'))
print('-' * 88)
for layer, how, a, n, na, nb, meaning in ladder:
    print('%-13s %-5s %8.4f %7d %7d %7d  %s' % (layer, how, a, n, na, nb, meaning))

#  Automatic reading of the ladder, on the `z` rows (the reported statistic).
#  MIN_EFFECT guards against reading noise as a direction: an AUROC of 0.4917 is not
#  "the low side", it is null. A first version of this cell omitted the guard and
#  announced RANK-RELATIVITY from 0.4896 vs 0.4917, which are both nothing.
MIN_EFFECT = 0.02        # |AUROC - 0.5| below this counts as no effect
side = {}
for layer, how, a, *_ in ladder:
    if how == 'z' and np.isfinite(a):
        if abs(a - 0.5) < MIN_EFFECT:
            side[layer[2:]] = 'null'
        else:
            side[layer[2:]] = 'low' if a < 0.5 else 'high'
print()
print('sides (z statistic, |AUROC-0.5| >= %.2f to count):' % MIN_EFFECT, side)
print()
print('HOW TO READ IT')
e, p, m = side.get('expression'), side.get('penalty'), side.get('MetChange')
if 'null' in (e, p):
    print('  layer 1 and/or layer 2 show NO effect on the z statistic, so neither supports nor')
    print('    refutes rank-relativity. Use the per-gene cell below instead: the decisive')
    print('    quantity is corr(d_expr, d_penalty) across genes, where ~-0.92 means the')
    print('    ordinary within-sample relation holds and rank and level have NOT come apart.')
elif e and p:
    if e == p:
        print('  layers 1 and 2 are on the SAME side -> RANK-RELATIVITY. The producer genes moved')
        print('    one way in absolute level and the other way in within-sample rank, so the')
        print('    expression arm and MetChange were never measuring the same thing. Compare the')
        print('    raw and z rows for layer 1 to see how much of this is a global shift.')
    else:
        print('  layers 1 and 2 are on OPPOSITE sides -> no rank-relativity; the penalties track')
        print('    the expression levels, so the sign is not set here.')
if p and m:
    if p == 'high' and m == 'low':
        print('  penalties got WORSE while the metabolites got EASIER -> the flip is in the LP.')
        print('    Expected cause: the LP minimises over routes, so the binding route is not the')
        print('    one made of the genes that deteriorated. Confirm by extracting the reactions')
        print('    carrying flux in each metabolite\'s optimal solution (needs Gurobi).')
    elif p == m:
        print('  penalty and MetChange agree -> the LP is faithful to its inputs here, so the')
        print('    flip must be downstream. Run the limma cell.')
print()
print('Re-run this cell with VARIANT = "mito_nohub" and "cyto_nohub": if the verdict differs')
print('between compartments, the explanation is compartment-specific and not a global artefact.')


MMP_down cases vs DMSO controls, tca_endpoints/all_nohub
173 producer genes, 24 metabolites

layer         stat     AUROC   nfeat   ncase   nctrl  higher value means
----------------------------------------------------------------------------------------
1 expression  raw     0.4055     168    2587    2895  higher = more transcript
1 expression  z       0.4896     168    2587    2895  higher = more transcript
2 penalty     raw     0.4969     168    2587    2895  higher = WORSE (less expressed)
2 penalty     z       0.4917     168    2587    2895  higher = WORSE (less expressed)
3 MetChange   raw     0.4522      20    2587    2895  higher = HARDER to produce
3 MetChange   z       0.3741      20    2587    2895  higher = HARDER to produce

sides (z statistic, |AUROC-0.5| >= 0.02 to count): {'expression': 'null', 'penalty': 'null', 'MetChange': 'low'}

HOW TO READ IT
  layer 1 and/or layer 2 show NO effect on the z statistic, so neither supports nor
    refutes rank-relativity. Use the pe

In [42]:
#  LAYER 4: limma. Does the fitted coefficient agree in sign with the raw pooled
#  difference of layer 3? If not, limma is where the sign is set - and that would mean the
#  pooled difference is confounded with batch/timepoint, which the limma design removes.
#
#  This is the comparison that cannot be done from the label matrix alone, because the
#  label matrix IS the limma output. Both quantities are needed side by side.

lbl_f = os.path.join(proj_dir_scratch, 'results/labels/HEPG2/metchange/rank/log2FC.csv')

lbl_cols = pl.scan_csv(lbl_f).collect_schema().names()
#  label columns are perturbation names "<drug>_<dose>_<time>", not sample ids
down_lbl_cols = [c for c in lbl_cols[1:] if c.rsplit('_', 2)[0] in down_drugs]
print('label columns matching an MMP_down drug: %d of %d' % (len(down_lbl_cols), len(lbl_cols) - 1))

LBL = (pl.scan_csv(lbl_f).select([lbl_cols[0]] + down_lbl_cols).collect()).to_pandas()
LBL = LBL.set_index(lbl_cols[0])

#  layer 3, raw: the pooled case-minus-control difference in MetChange score, per metabolite
mc_case = MC[[c for c in case_ids if c in MC.columns]].to_numpy(dtype=float)
mc_ctrl = MC[[c for c in ctrl_ids if c in MC.columns]].to_numpy(dtype=float)
raw_diff = pd.Series(np.nanmean(mc_case, axis=1) - np.nanmean(mc_ctrl, axis=1), index=MC.index)

#  layer 4: the limma coefficient, averaged over the MMP_down perturbations
limma_mean = LBL.mean(axis=1)

common = [m for m in mets if m in raw_diff.index and m in limma_mean.index]
print()
print('%-12s %-14s %14s %14s %7s' % ('metabolite', 'name', 'raw case-ctrl', 'limma mean', 'agree?'))
print('-' * 70)
for m in common:
    r, l = raw_diff[m], limma_mean[m]
    print('%-12s %-14s %14.4g %14.4g %7s'
          % (m, gem.metabolites.get_by_id(m).name, r, l,
             'yes' if np.sign(r) == np.sign(l) else 'NO <--'))

if common:
    rr, ll = raw_diff[common].to_numpy(float), limma_mean[common].to_numpy(float)
    ok = np.isfinite(rr) & np.isfinite(ll)
    agree = int((np.sign(rr[ok]) == np.sign(ll[ok])).sum())
    print()
    print('set mean, raw pooled difference : %+.4g   (positive = harder to produce)' % np.nanmean(rr))
    print('set mean, limma coefficient     : %+.4g' % np.nanmean(ll))
    print('metabolites agreeing in sign    : %d of %d' % (agree, int(ok.sum())))
    if ok.sum() > 2:
        print('correlation between the two     : %+.3f' % np.corrcoef(rr[ok], ll[ok])[0, 1])
    print()
    if agree < ok.sum() / 2:
        print('VERDICT: the raw and fitted differences DISAGREE -> the sign is set by limma,')
        print('         i.e. the pooled difference is confounded with batch/timepoint and the')
        print('         fitted coefficient is the one to trust. Inspect the batch balance above.')
    else:
        print('VERDICT: raw and fitted agree -> limma is NOT the source. The flip is upstream,')
        print('         at whichever layer the ladder above showed changing side.')


label columns matching an MMP_down drug: 605 of 15032

metabolite   name            raw case-ctrl     limma mean  agree?
----------------------------------------------------------------------
MAM01261c    acetyl-CoA             -1.403          1.026  NO <--
MAM01261g    acetyl-CoA              -1.57         0.9472  NO <--
MAM01261n    acetyl-CoA             -1.406          1.033  NO <--
MAM01261r    acetyl-CoA              -1.57         0.9472  NO <--
MAM01306c    AKG                    -66.75          34.34  NO <--
MAM01306e    AKG                    -65.99          35.13  NO <--
MAM01306m    AKG                    -64.97           42.9  NO <--
MAM01306x    AKG                    -66.93          34.89  NO <--
MAM01587c    citrate                -78.32          39.61  NO <--
MAM01587m    citrate                -78.68          42.22  NO <--
MAM01862c    fumarate               -112.1          34.51  NO <--
MAM01862e    fumarate               -112.4          35.31  NO <--
MAM01862m    fum

In [43]:
#  PER-GENE rank-relativity, and the contaminated subsystem set for contrast.
#
#  For each producer gene: did it fall in absolute expression, and did its penalty fall too?
#  Those two falling TOGETHER is rank-relativity - the gene lost expression but gained rank
#  because the rest of the transcriptome lost more.

cids = [c for c in case_ids if c in EXPR.columns and c in PNLT.columns]
tids = [c for c in ctrl_ids if c in EXPR.columns and c in PNLT.columns]

def delta(mat, rows, a, b):
    rows = [r for r in rows if r in mat.index]
    s = mat.loc[rows]
    return (s[a].to_numpy(float).mean(axis=1) - s[b].to_numpy(float).mean(axis=1)), rows

d_expr, rows_e = delta(EXPR, genes, cids, tids)
d_pnlt, rows_p = delta(PNLT, genes, cids, tids)
assert rows_e == rows_p, 'gene order diverged between the two matrices'

tab = pd.DataFrame({'gene': rows_e,
                    'symbol': [sym.get(g, g) for g in rows_e],
                    'd_expr': d_expr,
                    'd_penalty': d_pnlt})
tab['expr_down'] = tab.d_expr < 0
tab['penalty_down'] = tab.d_penalty < 0          # down penalty = BETTER rank
tab['rank_relativity'] = tab.expr_down & tab.penalty_down

n = len(tab)
print('producer genes of tca_endpoints/%s present in both matrices: %d' % (VARIANT, n))
print()
print('  expression DOWN and penalty DOWN (lost level, gained rank) : %3d  (%.0f%%)  <- rank-relativity'
      % (tab.rank_relativity.sum(), 100 * tab.rank_relativity.mean()))
print('  expression DOWN and penalty UP   (lost level, lost rank)   : %3d  (%.0f%%)'
      % (((tab.expr_down) & (~tab.penalty_down)).sum(), 100 * ((tab.expr_down) & (~tab.penalty_down)).mean()))
print('  expression UP   and penalty DOWN (gained level and rank)   : %3d  (%.0f%%)'
      % (((~tab.expr_down) & (tab.penalty_down)).sum(), 100 * ((~tab.expr_down) & (tab.penalty_down)).mean()))
print('  expression UP   and penalty UP   (gained level, lost rank) : %3d  (%.0f%%)'
      % (((~tab.expr_down) & (~tab.penalty_down)).sum(), 100 * ((~tab.expr_down) & (~tab.penalty_down)).mean()))
print()
print('  mean d_expr %+.4f   mean d_penalty %+.4f   correlation %+.3f'
      % (tab.d_expr.mean(), tab.d_penalty.mean(),
         np.corrcoef(tab.d_expr, tab.d_penalty)[0, 1] if n > 2 else np.nan))
print()
print('  A correlation near -1 is the ordinary within-sample relation (more expression, less')
print('  penalty). Anything weaker means rank and level have come apart across samples.')
print()
print('largest absolute penalty changes:')
print(tab.reindex(tab.d_penalty.abs().sort_values(ascending=False).index).head(15).to_string(index=False))

#  The same two numbers for the contaminated subsystem gene set, for contrast
print()
print('=' * 78)
for lab, gset in [('producers (used here)', set(genes)),
                  ('tca gene subsystem set', SUB_SET),
                  ('subsystem MINUS producers', SUB_SET - set(genes))]:
    de, r1 = delta(EXPR, sorted(gset), cids, tids)
    dp, _ = delta(PNLT, sorted(gset), cids, tids)
    if len(r1) == 0:
        continue
    print('%-26s n=%4d   mean d_expr %+.4f   mean d_penalty %+.4f'
          % (lab, len(r1), de.mean(), dp.mean()))
print()
print('If the subsystem set and the producer set disagree in sign, the reported "expression')
print('says TCA genes go down" was a statement about LDHA/aldolases rather than the citrate cycle.')


producer genes of tca_endpoints/all_nohub present in both matrices: 168

  expression DOWN and penalty DOWN (lost level, gained rank) :  12  (7%)  <- rank-relativity
  expression DOWN and penalty UP   (lost level, lost rank)   :  92  (55%)
  expression UP   and penalty DOWN (gained level and rank)   :  61  (36%)
  expression UP   and penalty UP   (gained level, lost rank) :   3  (2%)

  mean d_expr -0.0437   mean d_penalty +0.0012   correlation -0.924

  A correlation near -1 is the ordinary within-sample relation (more expression, less
  penalty). Anything weaker means rank and level have come apart across samples.

largest absolute penalty changes:
           gene          symbol    d_expr  d_penalty  expr_down  penalty_down  rank_relativity
ENSG00000103066 ENSG00000103066  0.369496  -0.050988      False          True            False
ENSG00000152952 ENSG00000152952 -0.537680   0.044029       True         False            False
ENSG00000160446 ENSG00000160446  0.301202  -0.042904    

## What the ladder said, and the question it leaves

Observed (`tca_endpoints/all_nohub`, 2587 case vs 2895 control samples):

| layer | raw | z |
|---|---|---|
| 1 expression (168 producer genes) | **0.4055** | 0.4896 |
| 2 penalty (same genes) | 0.4969 | 0.4917 |
| 3 MetChange (20 metabolites) | 0.4522 | **0.3741** |

Two readings, neither of which is "the penalty inverts the sign":

**The rank transform ERASED the expression effect, it did not invert it.** The producer genes are
genuinely lower in MMP-down samples (raw 0.4055), but that disappears under within-sample
standardisation (z 0.4896) — so the drop is a **global** transcriptional shift, not something
specific to TCA producers. The penalty, being a within-sample rank, therefore correctly reports
nothing (0.4969 / 0.4917). This is the rank penalty doing its job, but the consequence is that
**MetChange cannot see overall transcriptional suppression at all** — only relative reallocation.

**The MetChange effect is almost entirely RELATIVE.** raw 0.4522 against z 0.3741: in absolute cost
the TCA endpoints barely moved, and the reported association lives in the within-sample
standardisation. So the finding is not "MMP-down drugs make TCA easier to produce" but
**"MMP-down drugs make TCA less penalised than the rest of the metabolome"**. Nothing got easier;
other things got harder. That statement contradicts nothing, which dissolves the original puzzle.

**What is still unexplained.** The producer-gene penalties are flat (0.49), so the TCA metabolites'
relative improvement cannot come from their own producers improving. The rank transform is
zero-sum — every gene that gains rank is matched by one that loses — so some *other* part of the
metabolic network must be losing. The two cells below find out which, and whether it is coherent
biology or just the most variable genes moving.


In [44]:
#  IS THE EFFECT RELATIVE? Compare the TCA endpoints against the WHOLE metabolome.
#
#  *** READ THE WARNING BEFORE THE VERDICT AT THE BOTTOM OF THIS CELL. ***
#  This cell uses the POOLED case-minus-control difference, and the limma cell showed that
#  quantity is confounded with batch/timepoint: raw and fitted differ in SIGN on 20 of 20
#  metabolites, correlation -0.869. So the verdict printed here answers the question with
#  the wrong quantity and should not be trusted on its own. The cell is kept because the
#  raw-versus-z contrast is still informative about how much of the effect is relative; the
#  trustworthy version of the same test is the closing-the-loop cell, which uses the limma
#  coefficients.

glob_mc = pd.Series(np.nanmean(MC.to_numpy(dtype=float), axis=0), index=MC.columns)
set_mc_raw, n_set = set_scores(MC, mets, how='raw')
set_mc_z, _ = set_scores(MC, mets, how='z')

rows = []
for lab, s in [('TCA endpoints, raw cost', set_mc_raw),
               ('WHOLE metabolome, raw cost', glob_mc),
               ('TCA endpoints, z (relative)', set_mc_z)]:
    a, na, nb = auroc(s.reindex(case_ids).dropna(), s.reindex(ctrl_ids).dropna())
    c = float(np.nanmean(s.reindex(case_ids).dropna()))
    t = float(np.nanmean(s.reindex(ctrl_ids).dropna()))
    rows.append((lab, a, c, t, c - t))

print('MMP_down vs DMSO, MetChange cost (higher = harder to produce)')
print()
print('%-30s %8s %12s %12s %12s' % ('', 'AUROC', 'mean case', 'mean ctrl', 'difference'))
print('-' * 78)
for lab, a, c, t, d in rows:
    print('%-30s %8.4f %12.5g %12.5g %+12.5g' % (lab, a, c, t, d))

gd = rows[1][4]
sd_ = rows[0][4]
print()
print('the test: does the WHOLE metabolome get harder while the TCA endpoints do not?')
if gd > 0 and abs(sd_) < abs(gd):
    print('  YES. global cost rose by %+.5g, the TCA set by only %+.5g.' % (gd, sd_))
    print('  -> the association is TCA being SPARED, not TCA improving. The correct wording is')
    print('     "less penalised than the rest of the metabolome", and nothing is contradicted.')
elif gd < 0 and sd_ < gd:
    print('  NO - the TCA set fell MORE than the global level (%+.5g vs %+.5g), so there is a' % (sd_, gd))
    print('  genuine absolute improvement to explain, not just a relative one.')
else:
    print('  neither pattern cleanly: global %+.5g, set %+.5g. Read the raw AUROCs above.' % (gd, sd_))
print()
print('*** CAVEAT: the two difference columns above are POOLED case-minus-control, which')
print('*** the limma cell shows is confounded with batch/timepoint (0 of 20 metabolites')
print('*** agree in sign, r = -0.869). Trust the closing-the-loop cell instead.')
print()
print('how much of the z effect is the global term? ratio of distances from 0.5:')
print('  raw %.4f -> |0.5-AUROC| = %.4f' % (rows[0][1], abs(0.5 - rows[0][1])))
print('  z   %.4f -> |0.5-AUROC| = %.4f' % (rows[2][1], abs(0.5 - rows[2][1])))


MMP_down vs DMSO, MetChange cost (higher = harder to produce)

                                  AUROC    mean case    mean ctrl   difference
------------------------------------------------------------------------------
TCA endpoints, raw cost          0.4522       7350.8       7425.5      -74.657
WHOLE metabolome, raw cost       0.4813       9108.4       9126.5      -18.112
TCA endpoints, z (relative)      0.3741     -0.37493     -0.36199    -0.012946

the test: does the WHOLE metabolome get harder while the TCA endpoints do not?
  NO - the TCA set fell MORE than the global level (-74.657 vs -18.112), so there is a
  genuine absolute improvement to explain, not just a relative one.

*** CAVEAT: the two difference columns above are POOLED case-minus-control, which
*** the limma cell shows is confounded with batch/timepoint (0 of 20 metabolites
*** agree in sign, r = -0.869). Trust the closing-the-loop cell instead.

how much of the z effect is the global term? ratio of distances from 

In [45]:
#  WHO LOSES RANK? The rank transform is zero-sum: p = 1 - rank/n reassigns a fixed
#  distribution, so every gene gaining rank is matched by one losing it. If the TCA
#  producers are flat, something else moved. Is the loser a coherent pathway (biology) or
#  just the most variable genes (noise)?

#  per-gene penalty change, over the whole transcriptome
pc = PNLT[cids].to_numpy(dtype=float).mean(axis=1)
pt = PNLT[tids].to_numpy(dtype=float).mean(axis=1)
dpen = pd.Series(pc - pt, index=PNLT.index)          # > 0 = penalty WORSE in cases
print('penalty change across all %d genes: mean %+.5f (must be ~0 if the transform is zero-sum)'
      % (len(dpen), dpen.mean()))
print('  genes worse: %d   better: %d' % (int((dpen > 0).sum()), int((dpen < 0).sum())))
print()

#  compare the TCA producers against every other subsystem's GPR genes
sub_genes = defaultdict(set)
for r in gem.reactions:
    if r.subsystem:
        sub_genes[r.subsystem] |= {g.id for g in r.genes}

tca_prod = set(genes)
out = []
for sub, gs in sub_genes.items():
    idx = [g for g in gs if g in dpen.index]
    if len(idx) < 15:
        continue
    out.append((sub, len(idx), float(dpen.loc[idx].mean())))
out.sort(key=lambda x: -x[2])

tp = [g for g in tca_prod if g in dpen.index]
print('mean penalty change by subsystem (positive = that subsystem got WORSE in MMP_down)')
print('TCA producers used here: %+.5f  (n=%d)' % (dpen.loc[tp].mean(), len(tp)))
print()
print('%-58s %5s %10s' % ('subsystem', 'n', 'd_penalty'))
print('-' * 76)
for sub, n_, d in out[:12]:
    print('%-58s %5d %+10.5f' % (sub[:58], n_, d))
print('   ...')
for sub, n_, d in out[-8:]:
    print('%-58s %5d %+10.5f' % (sub[:58], n_, d))
print()
rank_of_tca = sum(1 for _, _, d in out if d > dpen.loc[tp].mean())
print('the TCA producers sit at position %d of %d subsystems, i.e. the %.0fth percentile'
      % (rank_of_tca + 1, len(out), 100 * (1 - rank_of_tca / max(len(out), 1))))
print()
print('READ: if the subsystems that got WORSE are a coherent group (e.g. biosynthesis,')
print('nucleotides, lipids) then the reallocation is biology and "TCA spared" is a real')
print('statement. If the ordering tracks nothing but gene count or variability, the z')
print('statistic is picking up which genes happen to move most, and the association is')
print('an artefact of the rank transform being zero-sum.')
print()
#  the noise check: does d_penalty track how variable a gene is, rather than its pathway?
var = pd.Series(PNLT.to_numpy(dtype=float).std(axis=1), index=PNLT.index)
ok = dpen.index.intersection(var.index)
print('correlation between |d_penalty| and a gene\'s penalty SD: %+.3f'
      % np.corrcoef(dpen.loc[ok].abs(), var.loc[ok])[0, 1])
print('  near +1 would mean the "winners and losers" are just the most variable genes.')


penalty change across all 2794 genes: mean +0.00108 (must be ~0 if the transform is zero-sum)
  genes worse: 1539   better: 1244

mean penalty change by subsystem (positive = that subsystem got WORSE in MMP_down)
TCA producers used here: +0.00115  (n=168)

subsystem                                                      n  d_penalty
----------------------------------------------------------------------------
Steroid metabolism                                            61   +0.01093
Fatty acid elongation (even-chain)                            15   +0.00926
Fatty acid elongation (odd-chain)                             15   +0.00926
Fatty acid biosynthesis (unsaturated)                         15   +0.00926
Androgen metabolism                                           22   +0.00913
Tryptophan metabolism                                         21   +0.00826
O-glycan metabolism                                           31   +0.00775
Fatty acid activation (cytosolic)                         

In [50]:
#  WHERE DOES THE DIMENSIONAL COLLAPSE HAPPEN: the penalty, or the LP?
#
#  Report 11 section 9.2 attributed it to the LP, on the grounds that the producer-average
#  (expression aggregated to the same metabolites, no LP) keeps 5.18 effective dimensions
#  while MetChange keeps 1.10. But the producer-average is built from expression log2FC and
#  NEVER PASSES THROUGH THE PENALTY:
#
#      MetChange        : absolute expr -> RANK PENALTY -> LP -> limma log2FC
#      producer-average : absolute expr ->              ->    -> limma log2FC -> mean
#
#  So that comparison eliminated metabolite-level aggregation, not the penalty. The rank
#  transform is an untested candidate. These three matrices are the actual chain, on the
#  same samples, so the step that collapses it is whichever one drops the dimensionality.
#
#  Note the rank penalty forces EVERY sample to the same marginal distribution, which is
#  exactly the kind of operation that can remove between-feature independence - so this is
#  a real possibility, not a formality.

def spectrum(mat, n_features=1500, seed=0, label=''):
    """PC1 share, participation ratio and median pairwise r of the FEATURE correlation matrix.

    Features are z-scored across samples first. No centring across features: that would
    subtract the shared component being measured (the bug that produced 9.55 instead of
    1.10 in the first pass at report 11 section 8).
    Non-finite and constant (zero-variance) rows are dropped: their correlation is undefined.
    """
    X = mat.to_numpy(dtype=float)
    n_total = X.shape[0]
    X = X[np.isfinite(X).all(axis=1)]
    n_finite = X.shape[0]

    sd = X.std(axis=1)
    X = X[sd > 1e-8 * max(np.abs(X).max(), 1.0)]   # drop constant / numerically-constant rows
    n_var = X.shape[0]

    Z = (X - X.mean(1, keepdims=True)) / X.std(1, keepdims=True)
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(Z), min(n_features, len(Z)), replace=False)
    C = np.corrcoef(Z[idx])
    assert np.isfinite(C).all(), 'non-finite correlations remain'

    print('  %-34s kept %d/%d rows (%d non-finite, %d constant)'
          % (label, n_var, n_total, n_total - n_finite, n_finite - n_var))
    ev = np.clip(np.linalg.eigvalsh(C)[::-1], 0, None)
    var = ev / ev.sum()
    off = C[np.triu_indices(len(idx), 1)]
    print('  %-34s %6d x %-5d  PC1 %5.1f%%  PR %7.2f  eig@90%% %5d  median r %+7.4f'
          % (label, X.shape[0], X.shape[1], 100 * var[0],
             ev.sum() ** 2 / (ev ** 2).sum(),
             int(np.searchsorted(np.cumsum(var), 0.90)) + 1, np.median(off)))
    return ev.sum() ** 2 / (ev ** 2).sum()

print('THE CHAIN, same samples throughout (absolute values, not log2FC)')
print()
pr_e = spectrum(EXPR, label='1 absolute expression (genes)')
pr_p = spectrum(PNLT, label='2 rank penalties (same genes)')
pr_m = spectrum(MC,   label='3 MetChange scores (metabolites)')

print()
print('drop from expression to penalty : %7.2f -> %7.2f   (factor %.1f)' % (pr_e, pr_p, pr_e / max(pr_p, 1e-9)))
print('drop from penalty to LP         : %7.2f -> %7.2f   (factor %.1f)' % (pr_p, pr_m, pr_p / max(pr_m, 1e-9)))
print()
if pr_p < 0.3 * pr_e and pr_m > 0.5 * pr_p:
    print('VERDICT: the RANK PENALTY collapses it, not the LP. Report 11 section 9.2 is wrong')
    print('  and must be corrected: the producer-average comparison never tested the penalty.')
    print('  Implication: a different penalty transform could change the representation, and')
    print('  the saturating mixture penalty would likely be WORSE (it compresses the middle of')
    print('  the distribution where the rank penalty is uniform). The smoother `graded_0` /')
    print('  `combined_0` options in create_penalties.py::make_penalties are the ones to try.')
elif pr_m < 0.3 * pr_p:
    print('VERDICT: the LP collapses it, as report 11 section 9.2 claimed - but now actually')
    print('  tested. The penalty passes its dimensionality through and the LP destroys it,')
    print('  so no penalty transform will rescue the representation and the problem is')
    print('  structural to taking a per-metabolite minimum over shared routes.')
else:
    print('VERDICT: neither step dominates - the collapse is gradual. Read the two factors above.')
print()
print('CAVEAT: absolute matrices here, where the reported dimensionality figures were on')
print('log2FC. A per-sample rank transform makes every sample marginally identical, so its')
print('effect on an absolute matrix and on a difference matrix need not be the same. If this')
print('cell and report 11 section 8 disagree, that difference is itself the finding.')


THE CHAIN, same samples throughout (absolute values, not log2FC)

  1 absolute expression (genes)      kept 20131/20587 rows (0 non-finite, 456 constant)
  1 absolute expression (genes)       20131 x 5482   PC1  20.2%  PR   10.69  eig@90%    38  median r +0.0149
  2 rank penalties (same genes)      kept 2783/2794 rows (0 non-finite, 11 constant)
  2 rank penalties (same genes)        2783 x 5482   PC1  21.7%  PR    9.70  eig@90%    35  median r +0.0084
  3 MetChange scores (metabolites)   kept 5700/5707 rows (7 non-finite, 0 constant)
  3 MetChange scores (metabolites)     5700 x 5482   PC1  95.2%  PR    1.10  eig@90%     1  median r +0.9733

drop from expression to penalty :   10.69 ->    9.70   (factor 1.1)
drop from penalty to LP         :    9.70 ->    1.10   (factor 8.8)

VERDICT: the LP collapses it, as report 11 section 9.2 claimed - but now actually
  tested. The penalty passes its dimensionality through and the LP destroys it,
  so no penalty transform will rescue the represen

In [51]:
#  SAVE. Everything above only printed to stdout, so nothing survives a kernel restart and
#  nothing can be cited from a report. This writes the durable artefacts plus a
#  run_info.json recording which files produced them - the provenance convention the rest
#  of the project uses, and the one whose absence has already cost this project a
#  withdrawn-then-reinstated result.
#
#  Defensive by design: it saves whatever cells you actually ran and says what is missing,
#  so a partial run still produces a usable record instead of a NameError.

import json
import subprocess
import datetime

out_dir = os.path.join(proj_dir_scratch, 'results/mc_phenotype_assoc/HEPG2/tca_sign_diagnostic')
os.makedirs(out_dir, exist_ok=True)
saved, missing = [], []


def save(name, obj, how='tsv'):
    if obj is None:
        missing.append(name)
        return
    path = os.path.join(out_dir, name)
    if how == 'tsv':
        obj.to_csv(path, sep='\t', index=False)
    else:
        with open(path, 'w') as fh:
            json.dump(obj, fh, indent=2, default=str)
    saved.append('%-34s %s' % (name, '%.1f kB' % (os.path.getsize(path) / 1024)))


g = globals()

#  the ladder (cell: layers 1-3)
save('ladder.tsv',
     pd.DataFrame(g['ladder'], columns=['layer', 'stat', 'auroc', 'n_features',
                                        'n_case', 'n_ctrl', 'higher_means'])
     if 'ladder' in g else None)

#  per-gene rank-relativity
save('per_gene_rank_relativity.tsv', g.get('tab'))

#  relative-vs-absolute MetChange
save('relative_vs_absolute.tsv',
     pd.DataFrame(g['rows'], columns=['quantity', 'auroc', 'mean_case', 'mean_ctrl', 'difference'])
     if 'rows' in g else None)

#  per-subsystem penalty change
save('penalty_change_by_subsystem.tsv',
     pd.DataFrame(g['out'], columns=['subsystem', 'n_genes', 'mean_d_penalty'])
     if 'out' in g else None)

#  the limma comparison, per metabolite
if 'raw_diff' in g and 'limma_mean' in g and 'common' in g:
    cm = g['common']
    save('limma_vs_raw.tsv', pd.DataFrame({
        'metabolite': cm,
        'name': [gem.metabolites.get_by_id(m).name for m in cm],
        'raw_case_minus_ctrl': g['raw_diff'][cm].to_numpy(float),
        'limma_mean_log2FC': g['limma_mean'][cm].to_numpy(float),
    }))
else:
    missing.append('limma_vs_raw.tsv')

#  whole-transcriptome penalty change, for reuse
save('penalty_change_per_gene.tsv',
     pd.DataFrame({'gene': g['dpen'].index,
                   'symbol': [sym.get(x, x) for x in g['dpen'].index],
                   'd_penalty': g['dpen'].to_numpy(float)})
     if 'dpen' in g else None)

#  provenance
try:
    commit = subprocess.check_output(['git', '-C', proj_dir, 'rev-parse', 'HEAD'],
                                     text=True).strip()
except Exception:
    commit = 'unavailable'

info = {
    'written': datetime.datetime.now().isoformat(timespec='seconds'),
    'git_commit_of_proj_dir': commit,
    'question': 'where is the MMP_down / TCA MetChange sign set: expression, penalty, LP or limma',
    'variant': g.get('VARIANT'),
    'inputs': {'expression': expr_f, 'penalties': pnlt_f, 'metchange_raw': mc_f,
               'limma_labels': g.get('lbl_f'), 'gem': gem_f,
               'phenotype_drugs': attene_drugs_f,
               'attene_datatable': attene_f},
    'n_case_samples': len(g.get('case_ids', [])),
    'n_ctrl_samples': len(g.get('ctrl_ids', [])),
    'n_case_drugs': int(len(g['attene_drugs_down'])) if 'attene_drugs_down' in g else None,
    'n_producer_genes': len(g.get('genes', [])),
    'n_metabolites_in_set': len(g.get('mets', [])),
    'chain_participation_ratio': {k: g.get(k) for k in ('pr_e', 'pr_p', 'pr_m')},
    'set_definition': {
        'tca_endpoints': g.get('TCA_ENDPOINTS'),
        'hub_rule': 'GEM reaction degree >= %s percentile, or name in CURRENCY_NAMES'
                    % g.get('HUB_PERCENTILE'),
        'note': 'MMP_down = Activity Summary == "active antagonist", aggregated by '
                'PRIORITY MMP_down > cytotox > MMP_up > inactive > incon; --strict-sep NOT used',
    },
}
save('run_info.json', info, how='json')

print('wrote to %s' % out_dir)
for s_ in saved:
    print('   %s' % s_)
if missing:
    print()
    print('NOT written (those cells were not run in this session):')
    for m in missing:
        print('   %s' % m)


wrote to /scratch/users/gsantamaria/test_large_files/metabognn_dev/results/mc_phenotype_assoc/HEPG2/tca_sign_diagnostic
   ladder.tsv                         0.5 kB
   per_gene_rank_relativity.tsv       15.0 kB
   relative_vs_absolute.tsv           0.3 kB
   penalty_change_by_subsystem.tsv    4.1 kB
   limma_vs_raw.tsv                   1.1 kB
   penalty_change_per_gene.tsv        145.4 kB
   run_info.json                      1.8 kB


In [52]:
#  CLOSING THE LOOP. limma says the TCA endpoints get HARDER in MMP_down drugs (+29.56),
#  yet the reported AUROC is 0.362, i.e. LOW. Both can be true only if the rest of the
#  metabolome gets harder by MORE, because `z` is the set's position relative to all
#  features within each perturbation. This cell checks that directly.
#
#  It is the quantity the pooled raw difference could not give, because that comparison is
#  confounded with batch/timepoint - the limma cell showed raw and fitted differ in sign on
#  20 of 20 metabolites, correlation -0.869.

LBL_ALL = (pl.scan_csv(lbl_f).select([lbl_cols[0]] + down_lbl_cols).collect()).to_pandas()
LBL_ALL = LBL_ALL.set_index(lbl_cols[0])

all_mean = LBL_ALL.to_numpy(dtype=float)
set_rows = [m for m in mets if m in LBL_ALL.index]
set_mean = LBL_ALL.loc[set_rows].to_numpy(dtype=float)

glob = float(np.nanmean(all_mean))
tca = float(np.nanmean(set_mean))
print('limma log2FC over the %d MMP_down perturbations (positive = HARDER to produce)' % LBL_ALL.shape[1])
print()
print('  TCA endpoints (%d features) : %+10.4f' % (len(set_rows), tca))
print('  WHOLE metabolome (%d)       : %+10.4f' % (LBL_ALL.shape[0], glob))
print('  difference                   : %+10.4f' % (tca - glob))
print()
if tca > 0 and glob > 0 and tca < glob:
    print('  CONFIRMED: everything gets harder, TCA least. "Relatively spared" is the right')
    print('  wording, the direction is IMPAIRMENT, and nothing contradicts the expression arm.')
elif tca > glob:
    print('  NOT confirmed: TCA moved MORE than the metabolome, so a low z cannot come from')
    print('  relative sparing. Re-derive the z statistic before trusting the AUROC.')
else:
    print('  neither: TCA %+.4f, global %+.4f. Read the signs before concluding.' % (tca, glob))

#  and the same split by perturbation, so it is not an artefact of averaging
per_pert = pd.DataFrame({
    'tca': np.nanmean(set_mean, axis=0),
    'global': np.nanmean(all_mean, axis=0)}, index=LBL_ALL.columns)
per_pert['tca_minus_global'] = per_pert.tca - per_pert['global']
frac = float((per_pert.tca_minus_global < 0).mean())
print()
print('  perturbations where TCA moved LESS than the metabolome: %.0f%% (%d of %d)'
      % (100 * frac, int((per_pert.tca_minus_global < 0).sum()), len(per_pert)))
print('  (a consistent effect should be well above 50%, not a few extreme perturbations)')
print()
print(per_pert.describe().loc[['mean', '50%', 'min', 'max']].to_string())


limma log2FC over the 605 MMP_down perturbations (positive = HARDER to produce)

  TCA endpoints (20 features) :   +29.5620
  WHOLE metabolome (5707)       :   +46.2816
  difference                   :   -16.7196

  CONFIRMED: everything gets harder, TCA least. "Relatively spared" is the right
  wording, the direction is IMPAIRMENT, and nothing contradicts the expression arm.

  perturbations where TCA moved LESS than the metabolome: 60% (361 of 605)
  (a consistent effect should be well above 50%, not a few extreme perturbations)

              tca       global  tca_minus_global
mean    29.562032    46.281321        -16.719290
50%     -0.164486    -3.989419         -9.288853
min  -1173.972298 -1050.434083       -529.114574
max   2598.889801  3128.004375        142.232601
